# EXP-016 | Stacking (LGB + CatBoost + XGBoost → 메타 러너)

단순 가중 평균 대신 메타 러너가 세 모델의 OOF 예측을 학습.

```
[LGB OOF]  ┐
[CAT OOF]  ├─→ Meta-Learner (LogisticRegression) → 최종 예측
[XGB OOF]  ┘
```

| 항목 | 내용 |
|---|---|
| Base 모델 | LGB(EXP-004) / CatBoost(EXP-011) / XGBoost(EXP-012) |
| Meta 러너 | LogisticRegression |
| CV | Stratified 5-Fold |
| 기준선 | EXP-015 앙상블-v2 |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 16
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 85)  /  X_test: (90067, 85)


## 2. Base 모델 파라미터

In [3]:
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    num_threads=-1,  # CPU 코어 전부 사용
    is_unbalance=True, learning_rate=0.05, num_leaves=127,
    min_child_samples=50, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=0.1,
)

CAT_PARAMS = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', random_seed=SEED,
    thread_count=-1,  # CPU 코어 전부 사용
    verbose=False, early_stopping_rounds=100,
    learning_rate=0.018758723768855998,
    depth=6,
    l2_leaf_reg=9.189608434163782,
    min_data_in_leaf=19,
    subsample=0.8170921295501483,
    colsample_bylevel=0.6936810336930781,
)

XGB_PARAMS = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist', random_state=SEED,
    n_jobs=-1,  # CPU 코어 전부 사용
    verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647,
    max_depth=4,
    min_child_weight=59,
    subsample=0.7663066457187595,
    colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928,
    reg_lambda=0.23932562420374562,
)

print('파라미터 설정 완료 (num_threads=-1 / thread_count=-1 / n_jobs=-1 → 전체 코어 사용)')

파라미터 설정 완료 (num_threads=-1 / thread_count=-1 / n_jobs=-1 → 전체 코어 사용)


## 3. Level-1: Base 모델 OOF 생성

각 base 모델을 5-Fold CV로 학습 → OOF 예측 수집.  
이 OOF가 메타 러너의 학습 피처가 됩니다.

In [4]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    ds_tr  = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    model  = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000,
                       valid_sets=[ds_val],
                       callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(period=-1)])
    oof_lgb[val_idx]  = model.predict(X_val)
    test_lgb         += model.predict(X_test) / N_FOLDS

auc_lgb = roc_auc_score(y_train, oof_lgb)
print(f'[1/3] LightGBM  OOF AUC: {auc_lgb:.5f}')

[1/3] LightGBM  OOF AUC: 0.73864


In [5]:
oof_cat  = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = CatBoostClassifier(**CAT_PARAMS)
    model.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    oof_cat[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_cat         += model.predict_proba(X_test)[:, 1] / N_FOLDS

auc_cat = roc_auc_score(y_train, oof_cat)
print(f'[2/3] CatBoost  OOF AUC: {auc_cat:.5f}')

[2/3] CatBoost  OOF AUC: 0.74015


In [6]:
oof_xgb  = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_xgb         += model.predict_proba(X_test)[:, 1] / N_FOLDS

auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'[3/3] XGBoost   OOF AUC: {auc_xgb:.5f}')
print(f'\nBase 모델 요약')
print(f'  LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}')

[3/3] XGBoost   OOF AUC: 0.74051

Base 모델 요약
  LGB=0.73864  CAT=0.74015  XGB=0.74051


## 4. Level-2: 메타 러너 학습

메타 피처 = `[oof_lgb, oof_cat, oof_xgb]` (3열)  
LogisticRegression이 세 모델의 예측을 최적 조합으로 학습.

In [7]:
# 메타 피처 구성
meta_train = np.stack([oof_lgb, oof_cat, oof_xgb], axis=1)   # (n_train, 3)
meta_test  = np.stack([test_lgb, test_cat, test_xgb], axis=1) # (n_test, 3)

# 메타 러너: LogisticRegression (C 탐색)
best_meta_auc, best_C, best_meta_pred = 0, 1.0, None
C_candidates = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

print('메타 러너 C 탐색 (5-Fold OOF)')
for C in C_candidates:
    meta_oof = np.zeros(len(meta_train))
    for tr_idx, val_idx in skf.split(meta_train, y_train):
        lr = LogisticRegression(C=C, max_iter=1000, random_state=SEED,
                                class_weight='balanced')
        lr.fit(meta_train[tr_idx], y_train.iloc[tr_idx])
        meta_oof[val_idx] = lr.predict_proba(meta_train[val_idx])[:, 1]
    auc = roc_auc_score(y_train, meta_oof)
    flag = ' ←' if auc > best_meta_auc else ''
    print(f'  C={C:<6}  OOF AUC={auc:.5f}{flag}')
    if auc > best_meta_auc:
        best_meta_auc, best_C = auc, C

print(f'\n최적 C={best_C}  OOF AUC={best_meta_auc:.5f}')

메타 러너 C 탐색 (5-Fold OOF)
  C=0.001   OOF AUC=0.74047 ←
  C=0.01    OOF AUC=0.74048 ←
  C=0.1     OOF AUC=0.74049 ←
  C=0.5     OOF AUC=0.74049
  C=1.0     OOF AUC=0.74048
  C=5.0     OOF AUC=0.74048
  C=10.0    OOF AUC=0.74048

최적 C=0.1  OOF AUC=0.74049


In [8]:
# 최적 C로 전체 train 재학습 → test 예측
meta_lr = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED,
                              class_weight='balanced')
meta_lr.fit(meta_train, y_train)
test_stack = meta_lr.predict_proba(meta_test)[:, 1]

# 메타 러너 계수 (각 모델 기여도)
coef = meta_lr.coef_[0]
coef_norm = np.abs(coef) / np.abs(coef).sum()
print('메타 러너 가중치 (절댓값 정규화)')
for name, w in zip(['LightGBM', 'CatBoost', 'XGBoost'], coef_norm):
    print(f'  {name:10s}: {w:.3f}')

# OOF AUC (메타 러너)
meta_oof_best = np.zeros(len(meta_train))
for tr_idx, val_idx in skf.split(meta_train, y_train):
    lr = LogisticRegression(C=best_C, max_iter=1000, random_state=SEED,
                             class_weight='balanced')
    lr.fit(meta_train[tr_idx], y_train.iloc[tr_idx])
    meta_oof_best[val_idx] = lr.predict_proba(meta_train[val_idx])[:, 1]

stacking_auc   = roc_auc_score(y_train, meta_oof_best)
stacking_prauc = average_precision_score(y_train, meta_oof_best)
stacking_f1    = f1_score(y_train, (meta_oof_best >= 0.5).astype(int))

print(f'\nStacking OOF ROC-AUC : {stacking_auc:.5f}')
print(f'Stacking OOF PR-AUC  : {stacking_prauc:.5f}')
print(f'Stacking OOF F1      : {stacking_f1:.5f}')
print(f'EXP-015 대비         : {stacking_auc - 0.74046:+.5f}  (앙상블-v2 기준)')

메타 러너 가중치 (절댓값 정규화)
  LightGBM  : 0.184
  CatBoost  : 0.380
  XGBoost   : 0.435

Stacking OOF ROC-AUC : 0.74049
Stacking OOF PR-AUC  : 0.45149
Stacking OOF F1      : 0.51714
EXP-015 대비         : +0.00003  (앙상블-v2 기준)


## 5. Submission 저장 & 실험 기록

In [9]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = test_stack
auc_str   = f'{stacking_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}')

저장: ../data/submissions/submission_exp016_조여진_07405.csv


In [10]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (meta_oof_best >= 0.5).astype(int)
params_str = (f'Base: LGB+CAT(EXP-011)+XGB(EXP-012) | '
              f'Meta: LogisticRegression(C={best_C}) | '
              f'LGB={auc_lgb:.5f} CAT={auc_cat:.5f} XGB={auc_xgb:.5f}')
NOTES    = f'2-Level Stacking. 메타 러너 LogisticRegression(C={best_C})'
INSIGHTS = (f'EXP-015 대비 {stacking_auc - 0.74046:+.5f} | '
            f'메타 가중치 LGB={coef_norm[0]:.3f} CAT={coef_norm[1]:.3f} XGB={coef_norm[2]:.3f}')

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Stacking(LGB+CAT+XGB→LR)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    stacking_auc, CV_STR, 'v1', X_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight + LR class_weight=balanced',
    'N', None, 'notebooks/12_stacking_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-016 기록 완료 (row 14)
